# T3.3 EIT-002 Environmental Footprint Simulation

**READ.ME**

This notebook calls the databases and project defined in EIT-003. The notebook is mainly a copy of EIT-001 as no example datasets for Simulation Configurations were shared yet.

- Setting the project based on EIT-003
- Read in Process Simulation Hub data
- Integration in foreground database as scenario 
- LCA results disaggregated in:
    - Total impact
    - Impact per tool / machining operation

### 1. Setting the project

In [ ]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash

13:53:55+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
config = Config()

In [3]:
bd.projects.set_current(config.PROJECT_NAME)

In [4]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.6
	BONSAI_V2.1.6 biosphere
	ecoinvent-3.10-biosphere
	reed

In [39]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

#### 2. Import Simulation Data

This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [9]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [17]:
simulation_data_path = config.REED_PATH/'Data'/"SimulationData"/"WWR"/"T2472"/"data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [20]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    simulation_data_path,
    tool_descriptions_path,debug=True
)
print(sensordata)


Total CSV files imported: 4
                                          energy_consumption  Power_Active  \
company event_id             Tool_Number                                     
wwr     OF10004_G_TWB_7SZ421 0.0               199943.657180   6813.293128   
                             81.0               79448.274472   8209.350979   
                             82.0               23343.902749   8310.724871   
                             100.0              67116.527595   8319.371254   
                             102.0             118242.215355   9726.310414   
...                                                      ...           ...   
                             2694.0              4718.325664  13096.354965   
                             2695.0               959.320473   9700.993548   
                             2696.0              1756.356816  13036.875333   
                             2722.0               351.314922  10199.465472   
                             3350.0

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [21]:
print(ambiguous_company_tools)

   company  Tool_Number                   op_type_candidates
0   goimek            2                 [AUXILIARY, MILLING]
1   goimek            6                 [AUXILIARY, MILLING]
2   goimek           30                 [AUXILIARY, MILLING]
3   goimek           31                   [CHAMFER, MILLING]
4   goimek           44                 [AUXILIARY, MILLING]
5   goimek           45                 [AUXILIARY, MILLING]
6  komatsu            3           [AUXILIARY, ROUGH MILLING]
7  komatsu           48  [AUXILIARY, MILLING, ROUGH MILLING]
8  komatsu           50        [ROUGH BORING, ROUGH MILLING]
9  komatsu           72      [FINISH MILLING, ROUGH MILLING]


In [22]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [23]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

### 3. Creating a scenario for Simulated Data integration

this section is missing a part about how to handle the processes that are not in hours?

In [24]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    get_product_id_for_activity,
)

Select which event you want to analyse

In [31]:
company = "wwr"
event_id = "OF10004_G_TWB_7SZ421"
product = "WWR total"
demand = 1

Start by selected the company and event_id where you want to swap in the sensordata

In [32]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
wwr     OF10004_G_TWB_7SZ421 boring, soraluce                   39487.347249   
                             drilling, soraluce                 54469.954977   
                             generic machining, soraluce       551951.801293   
                             milling, soraluce                 550725.038966   

                                                              hours  \
company event_id             reed tool                                
wwr     OF10004_G_TWB_7SZ421 boring, soraluce              3.190278   
                             drilling, soraluce            4.410833   
                             generic machining, soraluce  65.591389   
                             milling, soraluce            51.766944   

                                                          Power_Active  
company event_id   


Find all processes present in the sensordata in the reed-database

In [33]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'boring, soraluce': [],
 'drilling, soraluce': ['drilling, soraluce' (kilogram, DE, None),
  'drilling, soraluce' (hour, GB, None),
  'drilling, soraluce' (kilogram, ES, None),
  'drilling, soraluce' (kilogram, GB, None),
  'drilling, soraluce' (hour, ES, None),
  'drilling, soraluce' (hour, DE, None)],
 'generic machining, soraluce': ['generic machining, soraluce' (kilogram, GB, None),
  'generic machining, soraluce' (kilogram, ES, None),
  'generic machining, soraluce' (hour, DE, None),
  'generic machining, soraluce' (kilogram, DE, None),
  'generic machining, soraluce' (hour, ES, None),
  'generic machining, soraluce' (hour, GB, None)],
 'milling, soraluce': ['milling, soraluce' (hour, GB, None),
  'milling, soraluce' (hour, ES, None),
  'milling, soraluce' (hour, DE, None),
  'milling, soraluce' (kilogram, DE, None),
  'milling, soraluce' (kilogram, GB, None),
  'milling, soraluce' (kilogram, ES, None)]}

Find subactivities called A3 for the product under study

In [34]:
product_activity = reed.get(name=product)

# Find the A3 activity inside KOMATSU total
a3_matches = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)

print(a3_matches)
a3_activity = a3_matches[0]

['WWR A3: machining' (unit, DE, None)]


In [35]:
soraluce_matches = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)

print(soraluce_matches)

soraluce_activity = soraluce_matches[0]

['Soraluce machining, WWR' (unit, DE, None)]


In [40]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='production of electricity (market for)', location='DE', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [41]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)
scenarios


original  wwr_OF10004_G_TWB_7SZ421
product            activity                                              
310325776270831693 310338392070369280  0.000000                  0.044457
                   310338391986483200  0.000015                  0.030294
                   310338392020037632  0.000000                  0.038299

The input of products in found to consuming activities are found and used to build scenarios.

In [42]:
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
consumer_scenarios

,,original,wwr_OF10004_G_TWB_7SZ421
product,activity,,
310338391986483200,310338392154255361,0.000000,65.591389
310338392045203456,310338392154255361,30.633333,51.766944


In [43]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

In [44]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,wwr_OF10004_G_TWB_7SZ421
0,production of electricity (market for),TJ,"drilling, soraluce",hour,0.000000,0.044457
1,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000015,0.030294
2,production of electricity (market for),TJ,"milling, soraluce",hour,0.000000,0.038299
3,"generic machining, soraluce",hour,"Soraluce machining, WWR",unit,0.000000,65.591389
4,"milling, soraluce",hour,"Soraluce machining, WWR",unit,30.633333,51.766944


#### 4. Initial LCA scores for Footprint Simulation

LCA score with sensor data

In [45]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col=None,     # "original" or None (for defined scenario)
    amount=demand
)
score


319284.64418471605

LCA score without sensor data

In [46]:
product_reed = product_activity
lca = product_reed.lca(method=config.lcia_gwp100)
lca.score

39553.27339730245

In [47]:
wwr_total = reed.get(name="WWR total", location="DE", unit="unit")

In [48]:
bwa.print_recursive_calculation(wwr_total,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0.001)

Fraction of score | Absolute score | Amount | Activity
0001 | 3.955e+04 |     1 | 'WWR total' (unit, DE, None)
  0.996 | 3.94e+04 |     1 | 'WWR A1: material inputs' (unit, DE, None)
  0.0131 | 518.9 |     1 | 'WWR A2: transport' (unit, DE, None)
  0.0228 |   901 |     1 | 'WWR A3: machining' (unit, DE, None)
  -0.0321 | -1,269 | 0.825 | 'treatment of steel, industrial waste (market for)' (ton, DE, None)


### 4. Results preparation

#### 4.1 Functions for DPP alignment - **not needed in Footprint Simulation**

In [49]:
def lcia_score_for_named_input(
    parent_activity,
    name_fragment,
    method,
    scenario_df=None,
    scenario_col=None,
    amount=1.0,
):
    """
    Calculate LCIA score for direct inputs of parent_activity whose names contain name_fragment.
    Applies scenario_df if provided.
    """
    matches = [
        exc for exc in parent_activity.technosphere()
        if name_fragment.lower() in exc.input.get("name", "").lower()
    ]

    if not matches:
        raise ValueError(
            f"No input containing '{name_fragment}' found in {parent_activity.get('name')}"
        )

    total_score = 0

    for exc in matches:
        input_activity = exc.input
        input_amount = float(exc["amount"]) * amount

        if scenario_df is None:
            fu, data_objs, _ = bd.prepare_lca_inputs(
                {input_activity: input_amount},
                method=method,
            )

            lca = bc.LCA(fu, method=method, data_objs=data_objs)
            lca.lci()
            lca.lcia()

            total_score += lca.score

        else:
            score, _ = lcia_for_activity_with_scenario(
                activity=input_activity,
                method=method,
                scenario_df=scenario_df,
                scenario_col=scenario_col,
                amount=input_amount,
            )

            total_score += score

    return total_score

In [50]:
score_A1 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A1",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A2 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A2",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A3 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A3",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

In [51]:
score_total = score_A1 + score_A2 + score_A3
score_total

320553.36869838554

#### 4.2 To be shared with REED's MES 

- Deliver total climate impacts for simulated piece
- Deliver impacts per machining operation

In [ ]:
activity = reed.get(name=product)
location = activity.get("location")
unit = activity.get("unit")

def make_pcf_entry(score, life_cycle_phase):
    return {
        "PcfCalculationMethods": [
            {
                "PcfCalculationMethod": "ISO 14044"
            }
        ],
        "PCF in kg CO2eq": round(score, 1),
        "ProcessName": f"{product} {life_cycle_phase}",
        "ReferenceValueForCalculation": unit,
        "QuantityOfMeasureForCalculation": demand,
        "LifeCyclePhases": [
            {
                "LifeCyclePhase": life_cycle_phase
            }
        ],
        "ExplanatoryStatement": (
            "Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product."
        ),
        "GoodsHandoverAddress": {
            "Country": location
        },
        "PublicationDate": date.today().isoformat(),
        # "ExpirationDate": "2027-03-06"
    }

dpp_output = {
    "DPP": [
        {
            "company": company,
            "event_id": event_id,
            "product": product,
            "carbonFootprint": {
                "ProductCarbonFootprints": [
                    make_pcf_entry(score_A1, "A1"),
                    make_pcf_entry(score_A2, "A2"),
                    make_pcf_entry(score_A3, "A3"),
                    make_pcf_entry(score_A1 + score_A2 + score_A3, "A1-A3")  # optional total
                ]
            }
        }
    ]
}

dpp_output

{'DPP': [{'company': 'wwr',
   'event_id': 'OF10004_G_TWB_7SZ421',
   'product': 'WWR total',
   'carbonFootprint': {'ProductCarbonFootprints': [{'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PCF in kg CO2eq': 39402.1,
      'ProcessName': 'WWR total A1',
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePhases': [{'LifeCyclePhase': 'A1'}],
      'ExplanatoryStatement': 'Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product.',
      'GoodsHandoverAddress': {'Country': 'DE'},
      'PublicationDate': '2026-05-27'},
     {'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PCF in kg CO2eq': 518.9,
      'ProcessName': 'WWR total A2',
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePhases': [{'Lif